🔹 Step 1: Load + Clean Raw Data

In [3]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\G_BOOTS\OneDrive\Desktop\heritage flowers\SALES REREPORT-by 31st march.xlsx"

# Load March sheet
df = pd.read_excel(file_path, sheet_name="MARCH")

# Clean column names
df.columns = df.columns.str.strip()

🔹 Step 2: Rename Columns to Your Schema

In [4]:
df = df.rename(columns={
    "DATE": "Date",
    "CLIENT": "Customer",
    "Invoice to": "Invoice_To",
    "INVOICE NO.": "Invoice_No",
    "NO. STEMS": "Quantity_Sold",
    "NO. BOXES": "No_Boxes",
    "VALUE (USD)": "USD_Value",
    "VALUE(EUR)": "EUR_Value",
    "DROP POINT": "Drop_Off_Point",
    "DESTINATION": "Destination"
})

🔹 Step 3: Clean Currency Columns

In [6]:
# Remove symbols and clean
df["USD_Value"] = df["USD_Value"].astype(str).str.replace(r"[$, ]", "", regex=True)
df["EUR_Value"] = df["EUR_Value"].astype(str).str.replace(r"[€, ]", "", regex=True)

# Convert to numeric
df["USD_Value"] = pd.to_numeric(df["USD_Value"], errors="coerce")
df["EUR_Value"] = pd.to_numeric(df["EUR_Value"], errors="coerce")

🔹 Step 4: Create Currency + Total_Sales

In [7]:
def get_currency(row):
    if pd.notna(row["USD_Value"]) and row["USD_Value"] > 0:
        return "USD"
    elif pd.notna(row["EUR_Value"]) and row["EUR_Value"] > 0:
        return "EUR"
    else:
        return np.nan

def get_total_sales(row):
    if row["Currency"] == "USD":
        return row["USD_Value"]
    elif row["Currency"] == "EUR":
        return row["EUR_Value"]
    else:
        return 0

df["Currency"] = df.apply(get_currency, axis=1)
df["Total_Sales"] = df.apply(get_total_sales, axis=1)

🔹 Step 5: Add Missing Required Columns

In [8]:
# Market (derived)
df["Market"] = df["Currency"].apply(lambda x: "Local" if x == "KES" else "Export")

# Exchange rates (you can update these dynamically later)
exchange_rates = {
    "USD": 130,
    "EUR": 140,
    "KES": 1
}

df["Exchange_Rate_to_KES"] = df["Currency"].map(exchange_rates)

# Total in KES
df["Total_Sales_KES"] = df["Total_Sales"] * df["Exchange_Rate_to_KES"]

# Unit price
df["Unit_Price"] = df["Total_Sales"] / df["Quantity_Sold"]

🔹 Step 6: Select Final Columns

In [11]:
df_final = df[[
    "Date", "Customer", "Invoice_To", "Invoice_No",
    "Market", "Drop_Off_Point", "Currency",
    "Quantity_Sold", "No_Boxes", "Unit_Price",
    "Total_Sales", "Exchange_Rate_to_KES", "Total_Sales_KES"
]].copy()

🔹 Step 7: Final Cleaning

In [ ]:
# Fix dates
df_final.loc[:, "Date"] = pd.to_datetime(
    df_final["Date"].astype(str).str.strip(),
    dayfirst=True,
    errors="coerce"
)
# Convert numerics
numeric_cols = [
    "Quantity_Sold", "No_Boxes", "Unit_Price",
    "Total_Sales", "Exchange_Rate_to_KES", "Total_Sales_KES"
]

for col in numeric_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

# Drop junk rows (like "no shipment")
df_final = df_final.dropna(subset=["Invoice_No", "Quantity_Sold"], how="all")

🔹 Step 8: Save Clean Data

In [13]:
output_path = r"C:\Users\G_BOOTS\OneDrive\Desktop\heritage flowers\clean_march_sales.csv"
df_final.to_csv(output_path, index=False)

print("✅ March sales cleaned successfully")

✅ March sales cleaned successfully
